In [18]:
!pip -q uninstall -y transformers tokenizers accelerate compressed-tensors llmcompressor || true
!pip -q install "transformers==4.57.3" "tokenizers==0.22.1" "accelerate==1.10.1" "datasets==4.4.1" "safetensors==0.7.0" "sentencepiece==0.2.1"
!pip -q install "compressed-tensors==0.13.0" "llmcompressor==0.9.0.1"


In [1]:
import os, shutil
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

캘리브레이션 데이터는 MANTA-1M 학습 데이터에서 무작위로 256개를 섞어서 바로 선택하고, 별도의 길이 분포나 구조적인 고려는 하지 않음.

양자화 전략에서는 attention 계열(q/k/v/o, self_attn)과 embed, lm_head, rope 관련 모듈만 제외하고, 나머지 모든 Linear 레이어(사실상 MLP 전체)를 W4A16 GPTQ로 양자화. MLP의 처음과 끝까지 전부 4bit로 내려가므로 속도와 메모리 이득은 가장 크지만, 반대로 정확도 손실 위험도 가장 큰 구조. 특히 첫 블록과 마지막 블록의 MLP는 모델 출력에 민감해서, 이 부분까지 4bit로 내려가면 성능이 흔들릴 수 있음.

In [ ]:
MODEL_ID = "/content/drive/MyDrive/Colab Notebooks/open/base_model"

WORK_DIR = "/content/work_submit"
MODEL_OUT = f"{WORK_DIR}/model"   # submit.zip 최상위는 반드시 model/

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512   # 고정 (요구사항)

SCHEME = "W4A16"            # vLLM int4 경로에서 가장 무난

# 4bit 적용 "대상"은 Linear를 쓰되,
# attention 관련 레이어는 ignore로 빼서 정확도를 방어한다.
TARGETS = ["Linear"]

# 모델마다 명칭이 조금씩 달라서 '키워드 ignore'를 넓게 잡음
# (이게 길이 늘리는 것보다 정확도에 영향 큼)
IGNORE_KEYWORDS = [
    "embed_tokens", "lm_head",
    "q_proj", "k_proj", "v_proj", "o_proj",          # attention proj
    "self_attn", "attention",                        # 일부 모델 명칭
    "rotary", "rope",                                # rope 관련
]

print("[INFO] loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("[INFO] loaded.")

# =========================
# Calibration dataset: 길이 대신 "다양성"
# =========================
print("[INFO] loading calibration dataset (shuffle/select)...")
ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))

def preprocess(example):
    text = tokenizer.apply_chat_template(
        example["conversations"],
        add_generation_prompt=True,
        tokenize=False
    )
    return {"text": text}

ds = ds.map(preprocess, remove_columns=[c for c in ds.column_names if c != "text"])
print("[INFO] calibration prepared.")

# =========================
# Build ignore list robustly: 실제 모듈명 기반 필터링
# - llmcompressor의 ignore는 보통 '모듈명' 매칭을 타므로
#   모델 모듈명을 훑어서 ignore 리스트를 만들어준다.
# =========================
print("[INFO] building ignore module list...")
ignore_modules = set()

for name, module in model.named_modules():
    low = name.lower()
    if any(k in low for k in IGNORE_KEYWORDS):
        ignore_modules.add(name)

ignore_modules = sorted(ignore_modules)
print(f"[INFO] ignore modules: {len(ignore_modules)}")

# =========================
# Quantize
# =========================
print(f"[INFO] GPTQ start: scheme={SCHEME}, n={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH}")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=ignore_modules,  # 키워드가 아니라 실제 모듈명 리스트로 전달
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] quantization done.")

# =========================
# Save HF standard (vLLM 호환)
# =========================
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(MODEL_OUT, exist_ok=True)

print("[INFO] saving model...")
model.save_pretrained(MODEL_OUT, save_compressed=True, safe_serialization=True)
tokenizer.save_pretrained(MODEL_OUT)

# base_model에 있던 부가 파일들 복사
for fname in ["chat_template.jinja", "generation_config.json", "special_tokens_map.json", "tokenizer_config.json"]:
    src = Path(MODEL_ID) / fname
    dst = Path(MODEL_OUT) / fname
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

# =========================
# Make submit.zip with EXACT structure:
# submit.zip
#  └─ model/
# =========================
zip_base = "/content/submit"
if Path(zip_base + ".zip").exists():
    os.remove(zip_base + ".zip")

shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=WORK_DIR,
    base_dir="model",
)

print("[DONE] created:", zip_base + ".zip")
print("Verify: zip top-level should contain only 'model/'")

[INFO] loading tokenizer/model...


`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] loaded.
[INFO] loading calibration dataset (shuffle/select)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

...

2026-02-05T17:21:54.269591+0000 | compress | METRIC - time 1.19s
2026-02-05T17:21:54.270981+0000 | compress | METRIC - error 7505.21
2026-02-05T17:21:54.272596+0000 | compress | METRIC - GPU 0 | usage: 7.67% | total memory: 16 GB
2026-02-05T17:21:54.274982+0000 | compress | METRIC - Compressed module size: 16.973824 MB
2026-02-05T17:21:54.277426+0000 | compress_modules | INFO - Quantizing model.layers.29.mlp.up_proj using 256 samples
2026-02-05T17:21:55.416131+0000 | compress | METRIC - time 1.14s
2026-02-05T17:21:55.417548+0000 | compress | METRIC - error 6510.08
2026-02-05T17:21:55.418823+0000 | compress | METRIC - GPU 0 | usage: 7.67% | total memory: 16 GB
2026-02-05T17:21:55.420267+0000 | compress | METRIC - Compressed module size: 16.973824 MB
2026-02-05T17:21:55.421693+0000 | compress_modules | INFO - Quantizing model.layers.29.mlp.down_proj using 256 samples
2026-02-05T17:21:57.681805+0000 | compress | METRIC - time 2.26s
2026-02-05T17:21:57.683548+0000 | compress | METRIC - err

(31/31): Propagating: 100%|██████████| 256/256 [00:00<00:00, 1013.41it/s]


2026-02-05T17:22:03.111250+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-05T17:22:03.149190+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[INFO] quantization done.
[INFO] saving model...
2026-02-05T17:22:03.193104+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 90it [00:07, 12.45it/s]


[DONE] created: /content/submit.zip
Verify: zip top-level should contain only 'model/'


개선시도1 >>> 오히려 baseline보다 더 점수떨어짐 속도가 더 느려졌기 때문인듯

지금은 attention을 다 빼서 속도 점수 챙기는데 초반 1,2층과 마지막 1,2층 MLP는 FP16으로 남겨두고 중간층 MLP만 4bit로 압축 이러면 정확도는 더 좋아지고, 속도는 거의 비슷한 경우가 많다고 함 (대부분 토큰 처리 비용은 중간층 누적에서 나오니까)

In [ ]:
# @title
MODEL_ID = "/content/drive/MyDrive/Colab Notebooks/open/base_model"

WORK_DIR = "/content/work_submit"
MODEL_OUT = f"{WORK_DIR}/model"   # submit.zip 최상위는 반드시 model/

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512   # 고정 (요구사항)

SCHEME = "W4A16"            # vLLM int4 경로에서 가장 무난

# 4bit 적용 "대상"은 Linear를 쓰되,
# attention 관련 레이어는 ignore로 빼서 정확도를 방어한다.
TARGETS = ["Linear"]

# 모델마다 명칭이 조금씩 달라서 '키워드 ignore'를 넓게 잡음
# (이게 길이 늘리는 것보다 정확도에 영향 큼)
IGNORE_KEYWORDS = [
    "embed_tokens", "lm_head",
    "q_proj", "k_proj", "v_proj", "o_proj",          # attention proj
    "self_attn", "attention",                        # 일부 모델 명칭
    "rotary", "rope",                                # rope 관련
]
# 예: layers.0.mlp, layers.1.mlp, layers.29.mlp, layers.30.mlp 제외
IGNORE_KEYWORDS += ["layers.0.mlp", "layers.1.mlp", "layers.29.mlp", "layers.30.mlp"]

print("[INFO] loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("[INFO] loaded.")

# =========================
# Calibration dataset: 길이 대신 "다양성"
# =========================
print("[INFO] loading calibration dataset (shuffle/select)...")
ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))

def preprocess(example):
    text = tokenizer.apply_chat_template(
        example["conversations"],
        add_generation_prompt=True,
        tokenize=False
    )
    return {"text": text}

ds = ds.map(preprocess, remove_columns=[c for c in ds.column_names if c != "text"])
print("[INFO] calibration prepared.")

# =========================
# Build ignore list robustly: 실제 모듈명 기반 필터링
# - llmcompressor의 ignore는 보통 '모듈명' 매칭을 타므로
#   모델 모듈명을 훑어서 ignore 리스트를 만들어준다.
# =========================
print("[INFO] building ignore module list...")
ignore_modules = set()

for name, module in model.named_modules():
    low = name.lower()
    if any(k in low for k in IGNORE_KEYWORDS):
        ignore_modules.add(name)

ignore_modules = sorted(ignore_modules)
print(f"[INFO] ignore modules: {len(ignore_modules)}")

# =========================
# Quantize
# =========================
print(f"[INFO] GPTQ start: scheme={SCHEME}, n={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH}")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=ignore_modules,  # 키워드가 아니라 실제 모듈명 리스트로 전달
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] quantization done.")

# =========================
# Save HF standard (vLLM 호환)
# =========================
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(MODEL_OUT, exist_ok=True)

print("[INFO] saving model...")
model.save_pretrained(MODEL_OUT, save_compressed=True, safe_serialization=True)
tokenizer.save_pretrained(MODEL_OUT)

# base_model에 있던 부가 파일들 복사
for fname in ["chat_template.jinja", "generation_config.json", "special_tokens_map.json", "tokenizer_config.json"]:
    src = Path(MODEL_ID) / fname
    dst = Path(MODEL_OUT) / fname
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

# =========================
# Make submit.zip with EXACT structure:
# submit.zip
#  └─ model/
# =========================
zip_base = "/content/submit_firstlast"
if Path(zip_base + ".zip").exists():
    os.remove(zip_base + ".zip")

shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=WORK_DIR,
    base_dir="model",
)

print("[DONE] created:", zip_base + ".zip")
print("Verify: zip top-level should contain only 'model/'")

개선시도2 >>> 속도는 비슷하게 유지됨.. 다만 딱히 정확도가 높아지진...

캘리브레이션 샘플을 뽑는 방식이 앞은“랜덤 256개”였다면, 텍스트 길이를 기준으로 샘플 분포를 의도적으로 맞춤. 짧은 문장, 중간 길이 문장, 긴 문장을 각각 일정 비율로 섞어서 256개를 구성. 이를 위해 짧은 샘플이 부족해지는 문제를 피하려고, 짧은 문장 후보 풀과 긴 문장 후보 풀을 따로 만들어서 균형 있게 뽑음.

GPTQ가 activation 분포에 매우 민감하기 때문에 캘리브레이션 데이터가 긴 문장 위주로 치우치면, 짧은 입력에서의 activation이 제대로 보정되지 않고, 반대로 짧은 문장만 많아도 long-context 쪽 오차가 커짐. 이 문제를 줄여서 레이어별 양자화 오차를 더 고르게 만드는 전략

단점은, 모델을 만드는 과정이 가장 오래 걸린다는 점이지만 이건 제출 후 평가 시간에는 전혀 영향을 주지 않아서 ㅇㅋ?

In [ ]:
# @title
import numpy as np

# =========================
# Config
# =========================
MODEL_ID = "/content/drive/MyDrive/Colab Notebooks/open/base_model"

WORK_DIR = "/content/work_submit"
MODEL_OUT = f"{WORK_DIR}/model"   # submit.zip 최상위는 반드시 model/

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512  # 고정

SCHEME = "W4A16"
TARGETS = ["Linear"]

# 후보풀 크기(샘플링 품질 vs 속도)
CAND_LONG_POOL  = 6000   # 긴 샘플 후보
CAND_SHORT_POOL = 6000   # 짧은 샘플 후보(따로 확보)
SHORT_MAXLEN_FOR_POOL = 160  # "짧다" 판단은 512기준이 아니라 이 값 기준으로 후보를 모음

np.random.seed(42)

# =========================
# Load model/tokenizer
# =========================
print("[INFO] loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("[INFO] loaded.")

# =========================
# Ignore keywords
# =========================
IGNORE_KEYWORDS = [
    "embed_tokens", "lm_head",
    "q_proj", "k_proj", "v_proj", "o_proj",
    "self_attn", "attention",
    "rotary", "rope",
]
print("[INFO] ignore keywords:", IGNORE_KEYWORDS[-6:])

# =========================
# Build calibration dataset (length-balanced, practical)
# - short 후보를 따로 확보해서 버킷이 비는 문제 해결
# =========================
print("[INFO] loading dataset...")
raw_ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)

def build_text(example):
    return tokenizer.apply_chat_template(
        example["conversations"],
        add_generation_prompt=True,
        tokenize=False
    )

def tok_len(text, max_len):
    return len(tokenizer(text, truncation=True, max_length=max_len, add_special_tokens=False)["input_ids"])

print("[INFO] building candidate pools...")

# 1) long/mid 후보: 그냥 셔플해서 뽑기
cand_long = raw_ds.shuffle(seed=42).select(range(CAND_LONG_POOL))
cand_long_texts = [build_text(cand_long[i]) for i in range(len(cand_long))]
cand_long_lens512 = np.array([tok_len(t, MAX_SEQUENCE_LENGTH) for t in cand_long_texts])

# 2) short 후보: 160 토큰 컷으로 "짧은 샘플"이 많이 섞이도록 별도 풀 구성
cand_short = raw_ds.shuffle(seed=777).select(range(CAND_SHORT_POOL))
cand_short_texts = [build_text(cand_short[i]) for i in range(len(cand_short))]
cand_short_lens160 = np.array([tok_len(t, SHORT_MAXLEN_FOR_POOL) for t in cand_short_texts])

short_pool_idx = np.where(cand_short_lens160 <= SHORT_MAXLEN_FOR_POOL)[0]

# 버킷 정의(512 기준)
short_idx = np.where(cand_long_lens512 <= 128)[0]
mid_idx   = np.where((cand_long_lens512 > 128) & (cand_long_lens512 <= 256))[0]
long_idx  = np.where(cand_long_lens512 > 256)[0]

print(f"[INFO] long-pool short/mid/long: {len(short_idx)}, {len(mid_idx)}, {len(long_idx)}")
print(f"[INFO] short-pool candidates (<= {SHORT_MAXLEN_FOR_POOL} toks): {len(short_pool_idx)}")

per_bucket = NUM_CALIBRATION_SAMPLES // 3
need_long = NUM_CALIBRATION_SAMPLES - 2 * per_bucket

def sample_from(indices, n):
    if len(indices) == 0:
        return np.array([], dtype=int)
    if len(indices) <= n:
        return indices
    return np.random.choice(indices, size=n, replace=False)

# short는 "short 전용 풀"에서 뽑는다 (짧은 샘플 부족 문제 해결)
sel_short = sample_from(short_pool_idx, per_bucket)
# mid/long는 long 풀에서 뽑는다
sel_mid   = sample_from(mid_idx, per_bucket)
sel_long  = sample_from(long_idx, need_long)

# 선택한 인덱스들을 실제 dataset에서 가져오기
# short는 cand_short 기준, mid/long는 cand_long 기준이므로 따로 select 후 concat
ds_short = cand_short.select(sel_short.tolist()) if len(sel_short) else cand_short.select([])
ds_mid   = cand_long.select(sel_mid.tolist())     if len(sel_mid)   else cand_long.select([])
ds_long  = cand_long.select(sel_long.tolist())    if len(sel_long)  else cand_long.select([])

# datasets concatenate
from datasets import concatenate_datasets
ds = concatenate_datasets([ds_short, ds_mid, ds_long]).shuffle(seed=42)

# 혹시 수가 딱 256이 안 맞으면 보정
if len(ds) > NUM_CALIBRATION_SAMPLES:
    ds = ds.select(range(NUM_CALIBRATION_SAMPLES))
elif len(ds) < NUM_CALIBRATION_SAMPLES:
    # 부족하면 long 풀에서 추가로 채움
    remaining = NUM_CALIBRATION_SAMPLES - len(ds)
    extra = cand_long.select(np.random.choice(np.arange(len(cand_long)), size=remaining, replace=False).tolist())
    ds = concatenate_datasets([ds, extra]).shuffle(seed=42)

# 최종 text 컬럼 생성(map은 딱 1번만)
def preprocess(example):
    return {"text": build_text(example)}

ds = ds.map(preprocess, remove_columns=[c for c in ds.column_names if c != "text"])
print("[INFO] calibration dataset ready:", ds)

# =========================
# Build ignore module list from actual module names
# =========================
print("[INFO] building ignore module list...")
ignore_modules = []
for name, _ in model.named_modules():
    low = name.lower()
    if any(k in low for k in IGNORE_KEYWORDS):
        ignore_modules.append(name)

ignore_modules = sorted(set(ignore_modules))
print(f"[INFO] ignore modules: {len(ignore_modules)}")

# =========================
# Quantize
# =========================
print(f"[INFO] GPTQ start: scheme={SCHEME}, n={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH}")
recipe = [GPTQModifier(scheme=SCHEME, targets=TARGETS, ignore=ignore_modules)]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] quantization done.")

# =========================
# Save + zip (submit.zip)
# =========================
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(MODEL_OUT, exist_ok=True)

print("[INFO] saving model...")
model.save_pretrained(MODEL_OUT, save_compressed=True, safe_serialization=True)
tokenizer.save_pretrained(MODEL_OUT)

for fname in ["chat_template.jinja", "generation_config.json", "special_tokens_map.json", "tokenizer_config.json"]:
    src = Path(MODEL_ID) / fname
    dst = Path(MODEL_OUT) / fname
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

zip_base = "/content/submit2"
if Path(zip_base + ".zip").exists():
    os.remove(zip_base + ".zip")

shutil.make_archive(base_name=zip_base, format="zip", root_dir=WORK_DIR, base_dir="model")
print("[DONE] created:", zip_base + ".zip")
print("Verify: zip top-level should contain only 'model/'")

개선시도3

지금 attention을 전부 ignore. 하지만 ignore 키워드를 너무 넓게 잡으면 일부 Linear가 FP16으로 남아서 속도/메모리가 느려질 수 있음. 지금 IGNORE_KEYWORDS가 self_attn, attention 같은 너무 포괄적인 키워드가 포함돼서 모델에 따라선 mlp까지 걸려 FP16으로 남는 레이어가 생길 수도

개선: “정밀 ignore”로 좁혀서 정말 attention projection만 제외하자.

In [ ]:
MODEL_ID = "/content/drive/MyDrive/Colab Notebooks/open/base_model"

WORK_DIR = "/content/work_submit"
MODEL_OUT = f"{WORK_DIR}/model"   # submit.zip 최상위는 반드시 model/

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512   # 고정 (요구사항)

SCHEME = "W4A16"            # vLLM int4 경로에서 가장 무난

# 4bit 적용 "대상"은 Linear를 쓰되,
# attention 관련 레이어는 ignore로 빼서 정확도를 방어한다.
TARGETS = ["Linear"]

# 모델마다 명칭이 조금씩 달라서 '키워드 ignore'를 넓게 잡음
# (이게 길이 늘리는 것보다 정확도에 영향 큼)
IGNORE_KEYWORDS = [
    "embed_tokens", "lm_head",
    ".q_proj", ".k_proj", ".v_proj", ".o_proj",
    "rotary", "rope",
]

print("[INFO] loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("[INFO] loaded.")

# =========================
# Calibration dataset: 길이 대신 "다양성"
# =========================
print("[INFO] loading calibration dataset (shuffle/select)...")
ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))

def preprocess(example):
    text = tokenizer.apply_chat_template(
        example["conversations"],
        add_generation_prompt=True,
        tokenize=False
    )
    return {"text": text}

ds = ds.map(preprocess, remove_columns=[c for c in ds.column_names if c != "text"])
print("[INFO] calibration prepared.")

# =========================
# Build ignore list robustly: 실제 모듈명 기반 필터링
# - llmcompressor의 ignore는 보통 '모듈명' 매칭을 타므로
#   모델 모듈명을 훑어서 ignore 리스트를 만들어준다.
# =========================
print("[INFO] building ignore module list...")
ignore_modules = set()

for name, module in model.named_modules():
    low = name.lower()
    if any(k in low for k in IGNORE_KEYWORDS):
        ignore_modules.add(name)

ignore_modules = sorted(ignore_modules)
print(f"[INFO] ignore modules: {len(ignore_modules)}")

# =========================
# Quantize
# =========================
print(f"[INFO] GPTQ start: scheme={SCHEME}, n={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH}")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=ignore_modules,  # 키워드가 아니라 실제 모듈명 리스트로 전달
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] quantization done.")

# =========================
# Save HF standard (vLLM 호환)
# =========================
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(MODEL_OUT, exist_ok=True)

print("[INFO] saving model...")
model.save_pretrained(MODEL_OUT, save_compressed=True, safe_serialization=True)
tokenizer.save_pretrained(MODEL_OUT)

# base_model에 있던 부가 파일들 복사
for fname in ["chat_template.jinja", "generation_config.json", "special_tokens_map.json", "tokenizer_config.json"]:
    src = Path(MODEL_ID) / fname
    dst = Path(MODEL_OUT) / fname
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

# =========================
# Make submit.zip with EXACT structure:
# submit.zip
#  └─ model/
# =========================
zip_base = "/content/submit3"
if Path(zip_base + ".zip").exists():
    os.remove(zip_base + ".zip")

shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=WORK_DIR,
    base_dir="model",
)

print("[DONE] created:", zip_base + ".zip")
print("Verify: zip top-level should contain only 'model/'")

# 평가용 코드

In [5]:
from transformers import AutoTokenizer
from datasets import load_dataset

MODEL_ID = "/content/work_submit/model"

import torch
import math
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

def compute_ppl(model_dir, texts, max_len=512):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        local_files_only=True,
    )
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for t in texts:
            enc = tokenizer(
                t,
                return_tensors="pt",
                truncation=True,
                max_length=max_len,
            ).to(model.device)

            out = model(**enc, labels=enc["input_ids"])
            loss = out.loss

            n_tokens = enc["input_ids"].numel()
            total_loss += loss.item() * n_tokens
            total_tokens += n_tokens

    return math.exp(total_loss / total_tokens)


# 1. tokenizer 먼저 생성
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    local_files_only=True,
)

# 2. 그 다음 함수 정의
def build_text(ex):
    return tokenizer.apply_chat_template(
        ex["conversations"],
        add_generation_prompt=True,
        tokenize=False
    )

# 3. eval dataset
eval_ds = load_dataset(
    "LGAI-EXAONE/MANTA-1M",
    split="train[10000:10100]"
)

eval_texts = [build_text(eval_ds[i]) for i in range(len(eval_ds))]

ppl_1 = compute_ppl(MODEL_ID, eval_texts)
print(ppl_1)

The tokenizer you are loading from '/content/work_submit/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/work_submit/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Compressing model: 90it [00:00, 818.37it/s]


5.619389189316407


base: 5.456088036533331

best: 5.661500423906502 >>> 얘보다 확실히 높아야할듯

별로: 5.619777485612296

submit2: 5.619389189316407

In [6]:
import time

def benchmark_speed(model_dir, prompt, gen_tokens=128, repeat=5):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        local_files_only=True,
    )
    model.eval()

    enc = tokenizer(prompt, return_tensors="pt").to(model.device)

    # warmup
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=16)

    times = []
    with torch.no_grad():
        for _ in range(repeat):
            t0 = time.time()
            model.generate(**enc, max_new_tokens=gen_tokens)
            times.append(time.time() - t0)

    avg = sum(times) / len(times)
    return gen_tokens / avg


base tps: 446.2643447804525


best tps: 209.64123595144088 >> 210 언저리정도면 보통 10분 소요


별로 tps: 223.13556662755104


submit2: 212.53085189625082


In [10]:
prompt = "다음 문장을 요약해 주세요:\n" + eval_texts[0][:300]

print("tps:", benchmark_speed("/content/work_submit/model", prompt))

The tokenizer you are loading from '/content/work_submit/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Compressing model: 90it [00:00, 165.83it/s]


tps: 212.53085189625082
